In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    r"C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation"
)

KB_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "knowledge_base.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "knowledge"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

kb_df = pd.read_csv(KB_PATH)

print("Shape:", kb_df.shape)
print("Columns:", kb_df.columns.tolist())

display(kb_df.head())

Shape: (15979, 4)
Columns: ['prompt', 'context', 'response', 'source_dataset']


,prompt,context,response,source_dataset
0,What is (are) Monkeypox Virus Infections ?,NaN,Monkeypox is a rare viral disease. It occurs m...,MedQuAD
1,Do you have information about Vitamin K,NaN,Summary : Vitamins are substances that your bo...,MedQuAD
2,What is (are) phosphoribosylpyrophosphate synt...,NaN,Phosphoribosylpyrophosphate synthetase superac...,MedQuAD
3,What are the symptoms of Kallmann syndrome 6 ?,NaN,What are the signs and symptoms of Kallmann sy...,MedQuAD
4,Is Marfan syndrome inherited ?,NaN,How is Marfan syndrome inherited? Marfan syndr...,MedQuAD


In [ ]:
retrieval_df = kb_df.copy()

for column in ["prompt", "context", "response"]:
    retrieval_df[column] = (
        retrieval_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

retrieval_df["source_dataset"] = (
    retrieval_df["source_dataset"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
)

In [ ]:
def build_retrieval_text(row):

    parts = [
        f"Question: {row['prompt']}"
    ]

    if row["context"]:
        parts.append(
            f"Context: {row['context']}"
        )

    parts.append(
        f"Answer: {row['response']}"
    )

    return "\n\n".join(parts)


retrieval_df["retrieval_text"] = (
    retrieval_df.apply(
        build_retrieval_text,
        axis=1
    )
)

In [ ]:
retrieval_df.insert(
    0,
    "document_id",
    range(len(retrieval_df))
)

print(
    "Documents:",
    len(retrieval_df)
)

print(
    "Unique IDs:",
    retrieval_df["document_id"].nunique()
)

Documents: 15979
Unique IDs: 15979


In [ ]:
print("Empty retrieval documents:")

empty_count = (
    retrieval_df["retrieval_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(empty_count)
retrieval_df["text_length"] = (
    retrieval_df["retrieval_text"]
    .str.len()
)

display(
    retrieval_df["text_length"].describe()
)

Empty retrieval documents:
0


count    15979.000000
mean      1382.171725
std       1607.535678
min         71.000000
25%        578.500000
50%       1007.000000
75%       1721.000000
max      29109.000000
Name: text_length, dtype: float64

In [ ]:
print(
    retrieval_df["source_dataset"]
    .value_counts()
)

source_dataset
MedQuAD     14979
PubMedQA     1000
Name: count, dtype: int64


In [ ]:
print(
    "Duplicate retrieval texts:",
    retrieval_df["retrieval_text"].duplicated().sum()
)

print(
    "Duplicate prompt-response pairs:",
    retrieval_df[
        ["prompt", "response"]
    ].duplicated().sum()
)

Duplicate retrieval texts: 0
Duplicate prompt-response pairs: 0


In [ ]:
RETRIEVAL_PATH = (
    OUTPUT_DIR
    / "knowledge_base_retrieval.csv"
)

retrieval_df[
    [
        "document_id",
        "prompt",
        "context",
        "response",
        "source_dataset",
        "retrieval_text"
    ]
].to_csv(
    RETRIEVAL_PATH,
    index=False
)

print(
    "Saved:",
    RETRIEVAL_PATH
)

Saved: C:\Users\raich\Desktop\major\Healthcare_Dataset_Preparation\data\processed\knowledge\knowledge_base_retrieval.csv


In [ ]:
final_df = pd.read_csv(
    RETRIEVAL_PATH
)

print("Final shape:", final_df.shape)

print("\nColumns:")
print(final_df.columns.tolist())

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nSource distribution:")
print(
    final_df["source_dataset"].value_counts()
)

print("\nFirst document:")
print(
    final_df.iloc[0]["retrieval_text"]
)

Final shape: (15979, 6)

Columns:
['document_id', 'prompt', 'context', 'response', 'source_dataset', 'retrieval_text']

Missing values:
document_id           0
prompt                0
context           14979
response              0
source_dataset        0
retrieval_text        0
dtype: int64

Source distribution:
source_dataset
MedQuAD     14979
PubMedQA     1000
Name: count, dtype: int64

First document:
Question: What is (are) Monkeypox Virus Infections ?

Answer: Monkeypox is a rare viral disease. It occurs mostly in central and western Africa. Wild rodents and squirrels carry it, but it is called monkeypox because scientists saw it first in lab monkeys. In 2003, it was reported in prairie dogs and humans in the U.S.     Centers for Disease Control and Prevention
